# 广告投放问题

**类别：** 非线性

来源：[https://www.hexaly.com/templates/advertising-campaign-problem](https://www.hexaly.com/templates/advertising-campaign-problem)


## 问题描述

在**广告投放问题**中,一组广告可以在各个时间段上播出。每条广告在触达目标受众(例如特定年龄段)时会产生预期的营销收益。当一条广告被分配到某个时间段时,它会以给定的概率触达其目标受众。如果该广告在多个时间段播出,该概率会增大。目标是为每个时段分配一条广告,以使整个广告活动的总预期收益最大化。

该问题也被称为武器目标分配问题(Weapon Target Assignment Problem, WTA)。关于该问题的更多细节,请参阅[维基百科](https://en.wikipedia.org/wiki/Weapon_target_assignment_problem)。

	

### 学习要点

- 使用 OptAgent 的 `set` 建模分配给每个广告的时段
- 使用 `array` 和 `partition` 确保每个时段恰好分配给一条广告
- 使用 `prod` 和 `lambda_function` 构建每条广告的非线性预期收益


## 数据

本算例是对 [GitHub](https://github.com/tuliotoffolo/wta/tree/main/data) 上提供的武器目标分配算例的略微修改版本。数据格式如下:

- 第一行:时段数量,广告数量
- 接下来每一行对应一条广告的营销收益
- 之后的每一行包含:

- 时段的索引
- 广告的索引
- 该广告在该时段触达目标受众的概率


## 建模方法

广告投放问题的 OptAgent 模型使用集合决策变量。我们通过 `set` 为每条广告定义一个集合,其中包含分配给该广告的所有时段；再用 `array` 将这些集合组成一等数组,并施加 `partition` 约束,确保每个时段恰好被分配一条广告。

对于每条广告,我们用 `lambda_function` 表达各个已分配时段的未触达概率,再用集合聚合 `prod` 计算整场活动均未触达的概率。用 1 减去该乘积并乘以营销收益,即可得到广告的预期收益。最后对所有广告的预期收益求和并最大化。


## Python 实现


In [1]:
from optagent import ModelBuilder, solve

def read_instance(instance_filename):
    with open(instance_filename, encoding="utf-8") as f:
        lines = f.readlines()

    nb_slots = int(lines[0].split()[0])
    nb_ads = int(lines[0].split()[1])
    ad_profits = [int(lines[i]) for i in range(1, nb_ads + 1)]
    probabilities_data = [[0.0 for _ in range(nb_ads)] for _ in range(nb_slots)]
    for line in lines[nb_ads + 1:]:
        values = line.split()
        s = int(values[0])
        a = int(values[1])
        probabilities_data[s][a] = float(values[2])
    return nb_slots, nb_ads, ad_profits, probabilities_data


def solve_instance(ad_profits, probabilities_data, output_file=None, time_limit=60):
    nb_slots = len(probabilities_data)
    nb_ads = len(ad_profits)
    model = ModelBuilder()

    # assignments[a] is the set of slots assigned to ad a.
    assignments = [model.set(nb_slots, name=f"ad_{a}_slots") for a in range(nb_ads)]
    assignments_array = model.array(assignments)

    # Every slot must be assigned to exactly one ad.
    model.constraint(model.partition(assignments_array), name="unique_slot_assignment")

    # The empty product is 1, so an ad with no slots contributes zero profit.
    probabilities = model.array(probabilities_data)
    ad_profit_expressions = []
    for a in range(nb_ads):
        miss_probability = model.prod(
            assignments[a],
            model.lambda_function(
                lambda slot, ad=a: 1 - model.at(probabilities, slot, ad)
            ),
        )
        ad_profit_expressions.append(ad_profits[a] * (1 - miss_probability))
    total_profit = model.sum(*ad_profit_expressions)

    model.maximize(total_profit, name="total_expected_profit")
    solution = solve(model, time_limit_s=float(time_limit))

    lines = [
        f"Nb slots = {nb_slots}; Nb ads = {nb_ads}; "
        f"Total profit = {solution.objective_value}"
    ]
    for a, assignment in enumerate(assignments):
        assigned_slots = solution.variable_values[assignment.node_id]
        if assigned_slots:
            slots = ", ".join(f"slot {slot}" for slot in sorted(assigned_slots))
            lines.append(f"Ad {a} assigned to {slots}")

    result_text = "\n".join(lines)
    print(result_text)
    if output_file is not None:
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(result_text + "\n")
    return solution


def main(input_file, output_file=None, time_limit=60):
    _, _, ad_profits, probabilities_data = read_instance(input_file)
    return solve_instance(ad_profits, probabilities_data, output_file, time_limit)


# if __name__ == "__main__":
#     if len(sys.argv) < 2:
#         print("Usage: python advertising_campaign.py instance_file [output_file] [time_limit]")
#         sys.exit(1)
#     input_file = sys.argv[1]
#     output_file = sys.argv[2] if len(sys.argv) >= 3 else None
#     time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
#     main(input_file, output_file, time_limit)


## 运行实例

Notebook 直接调用 `main` 并显式传入实例路径。以下三个代码格相互独立，可以按需要单独运行；调整 `time_limit` 可以控制每个实例的求解时间。


In [2]:
from pathlib import Path

INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)

Instances: /Users/dongbox/work/opt-agent/examples/examples/hexaly/advertising_campaign_problem/instances


In [3]:
solution_50x100 = main(
    INSTANCE_DIR / "ad_campaign_50x100.txt",
    time_limit=1,
)


Starting OptAgent PORTFOLIO
Parameters: time_limit=1s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 1615.0390901901294
  improvements: initial=3 search=0
  evaluated: 0
  wall_time: 1.76711s
  termination: wall_time_exhausted


Nb slots = 50; Nb ads = 100; Total profit = 1615.0390901901294
Ad 1 assigned to slot 3
Ad 3 assigned to slot 43
Ad 6 assigned to slot 19
Ad 7 assigned to slot 7, slot 29
Ad 8 assigned to slot 5
Ad 10 assigned to slot 4
Ad 19 assigned to slot 6
Ad 21 assigned to slot 1, slot 9, slot 48
Ad 23 assigned to slot 42
Ad 25 assigned to slot 17
Ad 29 assigned to slot 18
Ad 33 assigned to slot 0, slot 40
Ad 34 assigned to slot 23
Ad 35 assigned to slot 27, slot 34
Ad 36 assigned to slot 28
Ad 37 assigned to slot 20, slot 24
Ad 41 assigned to slot 22
Ad 42 assigned to slot 25
Ad 44 assigned to slot 41
Ad 45 assigned to slot 8
Ad 48 assigned to slot 14
Ad 51 assigned to slot 10, slot 47
Ad 52 assigned to slot 26
Ad 54 assigned to slot 13
Ad 56 assigned to slot 31
Ad 59 assigned to slot 21
Ad 61 assigned to slot 16, slot 49
Ad 66 assigned to slot 45
Ad 68 assigned to slot 33
Ad 70 assigned to slot 15
Ad 71 assigned to slot 32
Ad 79 assigned to slot 44
Ad 82 assigned to slot 11
Ad 86 assigned to slo

In [4]:
solution_100x200 = main(
    INSTANCE_DIR / "ad_campaign_100x200.txt",
    time_limit=1,
)


Starting OptAgent PORTFOLIO
Parameters: time_limit=1s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 2412.744954065088
  improvements: initial=4 search=0
  evaluated: 0
  wall_time: 1.08623s
  termination: wall_time_exhausted


Nb slots = 100; Nb ads = 200; Total profit = 2412.744954065088
Ad 0 assigned to slot 80, slot 98
Ad 2 assigned to slot 60
Ad 3 assigned to slot 56, slot 61
Ad 4 assigned to slot 42
Ad 12 assigned to slot 63
Ad 21 assigned to slot 71, slot 84
Ad 23 assigned to slot 74
Ad 24 assigned to slot 3, slot 35, slot 87, slot 97
Ad 25 assigned to slot 25, slot 77
Ad 30 assigned to slot 1
Ad 31 assigned to slot 69
Ad 33 assigned to slot 28
Ad 34 assigned to slot 22
Ad 35 assigned to slot 27, slot 89
Ad 38 assigned to slot 18
Ad 39 assigned to slot 48
Ad 40 assigned to slot 47
Ad 44 assigned to slot 4, slot 6
Ad 45 assigned to slot 72
Ad 47 assigned to slot 45
Ad 48 assigned to slot 10
Ad 49 assigned to slot 70
Ad 53 assigned to slot 15, slot 39
Ad 62 assigned to slot 8
Ad 69 assigned to slot 54
Ad 81 assigned to slot 76
Ad 82 assigned to slot 90
Ad 83 assigned to slot 79
Ad 86 assigned to slot 37
Ad 87 assigned to slot 95
Ad 89 assigned to slot 67
Ad 95 assigned to slot 17
Ad 99 assigned to slot 6

In [5]:
solution_150x300 = main(
    INSTANCE_DIR / "ad_campaign_150x300.txt",
    time_limit=1,
)


Starting OptAgent PORTFOLIO
Parameters: time_limit=1s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 3462.387427687709
  improvements: initial=1 search=0
  evaluated: 0
  wall_time: 1.0838s
  termination: wall_time_exhausted


Nb slots = 150; Nb ads = 300; Total profit = 3462.387427687709
Ad 0 assigned to slot 80
Ad 6 assigned to slot 93
Ad 7 assigned to slot 85
Ad 9 assigned to slot 43, slot 45
Ad 14 assigned to slot 122
Ad 23 assigned to slot 25, slot 120
Ad 27 assigned to slot 7
Ad 28 assigned to slot 68
Ad 32 assigned to slot 3, slot 51, slot 114
Ad 33 assigned to slot 40
Ad 38 assigned to slot 49
Ad 42 assigned to slot 101
Ad 43 assigned to slot 21
Ad 45 assigned to slot 138
Ad 46 assigned to slot 124
Ad 49 assigned to slot 149
Ad 50 assigned to slot 2
Ad 52 assigned to slot 18
Ad 60 assigned to slot 12
Ad 61 assigned to slot 128, slot 135
Ad 63 assigned to slot 26, slot 82
Ad 64 assigned to slot 144
Ad 66 assigned to slot 97
Ad 68 assigned to slot 71
Ad 71 assigned to slot 130
Ad 74 assigned to slot 53, slot 74, slot 95
Ad 76 assigned to slot 91
Ad 77 assigned to slot 126
Ad 80 assigned to slot 16
Ad 81 assigned to slot 118
Ad 84 assigned to slot 111
Ad 91 assigned to slot 65
Ad 97 assigned to slot 13
